In [8]:
import pandas as pd
from pathlib import Path

processed_dir = Path("../data/processed/9558")

laps = pd.read_parquet(processed_dir / "laps.parquet")
stints = pd.read_parquet(processed_dir / "stints.parquet")

print("Laps:")
print(laps.shape)

print("\nStints:")
print(stints.shape)

Laps:
(962, 16)

Stints:
(65, 8)


In [9]:
print("Laps")
display(
    laps[["driver_number","lap_number","lap_duration","date_start"]].head(10)
)

print("Stints")
display(
    stints[["driver_number","stint_number",
            "lap_start",
            "lap_end",
            "compound",
            "tyre_age_at_start",]].head(10)
)

Laps


,driver_number,lap_number,lap_duration,date_start
0,1,1,97.211,2024-07-07 14:03:12.540000+00:00
1,1,2,91.773,2024-07-07 14:04:49.750000+00:00
2,1,3,92.200,2024-07-07 14:06:21.461000+00:00
3,1,4,91.933,2024-07-07 14:07:53.675000+00:00
4,1,5,91.778,2024-07-07 14:09:25.527000+00:00
5,1,6,91.711,2024-07-07 14:10:57.353000+00:00
6,1,7,91.935,2024-07-07 14:12:29.113000+00:00
7,1,8,92.038,2024-07-07 14:14:01.043000+00:00
8,1,9,91.883,2024-07-07 14:15:33.002000+00:00
9,1,10,91.661,2024-07-07 14:17:04.948000+00:00


Stints


,driver_number,stint_number,lap_start,lap_end,compound,tyre_age_at_start
0,1,1,1.0,26.0,MEDIUM,0
1,1,2,27.0,38.0,INTERMEDIATE,0
2,1,3,39.0,52.0,HARD,0
3,2,1,1.0,27.0,MEDIUM,0
4,2,2,28.0,38.0,INTERMEDIATE,0
5,2,3,39.0,52.0,SOFT,0
6,3,1,1.0,26.0,MEDIUM,0
7,3,2,27.0,37.0,INTERMEDIATE,0
8,3,3,38.0,51.0,SOFT,0
9,4,1,1.0,27.0,MEDIUM,0


In [10]:
ver_stints = stints[stints["driver_number"] == 1]

display(
    ver_stints[["stint_number","lap_start","lap_end","compound","tyre_age_at_start"]]
)

,stint_number,lap_start,lap_end,compound,tyre_age_at_start
0,1,1.0,26.0,MEDIUM,0
1,2,27.0,38.0,INTERMEDIATE,0
2,3,39.0,52.0,HARD,0


In [11]:
ver_laps = laps[laps["driver_number"] == 1].copy()

ver_laps["stint_number"] = pd.NA
ver_laps["compound"] = pd.NA
ver_laps["tyre_age"] = pd.NA

for _, stint in ver_stints.iterrows():
    mask = (
        (ver_laps["lap_number"] >= stint["lap_start"])
        & (ver_laps["lap_number"] <= stint["lap_end"])
    )

    ver_laps.loc[mask, "stint_number"] = stint["stint_number"]
    ver_laps.loc[mask, "compound"] = stint["compound"]

    ver_laps.loc[mask, "tyre_age"] = (
        ver_laps.loc[mask, "lap_number"] - stint["lap_start"] + stint["tyre_age_at_start"]
    )

display(
    ver_laps[[
        "lap_number",
        "lap_duration",
        "stint_number",
        "compound",
        "tyre_age"
    ]].head(30)
)

,lap_number,lap_duration,stint_number,compound,tyre_age
0,1,97.211,1,MEDIUM,0.0
1,2,91.773,1,MEDIUM,1.0
2,3,92.200,1,MEDIUM,2.0
3,4,91.933,1,MEDIUM,3.0
4,5,91.778,1,MEDIUM,4.0
5,6,91.711,1,MEDIUM,5.0
6,7,91.935,1,MEDIUM,6.0
7,8,92.038,1,MEDIUM,7.0
8,9,91.883,1,MEDIUM,8.0
9,10,91.661,1,MEDIUM,9.0


In [12]:
ver_laps["stint_number"] = pd.NA

for _, stint in ver_stints.iterrows():
    mask = (
        (ver_laps["lap_number"] >= stint["lap_start"]) &
        (ver_laps["lap_number"] <= stint["lap_end"])
    )
    ver_laps.loc[mask, "stint_number"] = stint["stint_number"]

display(
    ver_laps[
        ["lap_number", "lap_duration", "stint_number"]
    ].head(30)
)

,lap_number,lap_duration,stint_number
0,1,97.211,1
1,2,91.773,1
2,3,92.200,1
3,4,91.933,1
4,5,91.778,1
5,6,91.711,1
6,7,91.935,1
7,8,92.038,1
8,9,91.883,1
9,10,91.661,1


In [13]:
ver_laps["compound"] = pd.NA

for _, stint in ver_stints.iterrows():
    mask = (
        (ver_laps["lap_number"] >= stint["lap_start"]) &
        (ver_laps["lap_number"] <= stint["lap_end"])
    )

    ver_laps.loc[mask, "compound"] = stint["compound"]

display(
    ver_laps[
        [
            "lap_number",
            "lap_duration",
            "stint_number",
            "compound"
        ]
    ].head(30)
)

,lap_number,lap_duration,stint_number,compound
0,1,97.211,1,MEDIUM
1,2,91.773,1,MEDIUM
2,3,92.200,1,MEDIUM
3,4,91.933,1,MEDIUM
4,5,91.778,1,MEDIUM
5,6,91.711,1,MEDIUM
6,7,91.935,1,MEDIUM
7,8,92.038,1,MEDIUM
8,9,91.883,1,MEDIUM
9,10,91.661,1,MEDIUM


In [14]:
ver_laps["tyre_age"] = pd.NA

for _, stint in ver_stints.iterrows():
    mask = (
        (ver_laps["lap_number"] >= stint["lap_start"]) &
        (ver_laps["lap_number"] <= stint["lap_end"])
    )

    ver_laps.loc[mask, "tyre_age"] = (
        ver_laps.loc[mask, "lap_number"] - stint["lap_start"] + stint["tyre_age_at_start"]
    )

display(
    ver_laps[
        [
            "lap_number",
            "lap_duration",
            "stint_number",
            "compound",
            "tyre_age"
        ]
    ].head(30)
)

,lap_number,lap_duration,stint_number,compound,tyre_age
0,1,97.211,1,MEDIUM,0.0
1,2,91.773,1,MEDIUM,1.0
2,3,92.200,1,MEDIUM,2.0
3,4,91.933,1,MEDIUM,3.0
4,5,91.778,1,MEDIUM,4.0
5,6,91.711,1,MEDIUM,5.0
6,7,91.935,1,MEDIUM,6.0
7,8,92.038,1,MEDIUM,7.0
8,9,91.883,1,MEDIUM,8.0
9,10,91.661,1,MEDIUM,9.0


In [15]:
check_laps = ver_laps[
    ver_laps["lap_number"].isin(
        [1, 2, 25, 26, 27, 28, 37, 38, 39, 40, 51, 52]
    )
]

display(
    check_laps[
        [
            "lap_number",
            "lap_duration",
            "stint_number",
            "compound",
            "tyre_age"
        ]
    ]
)

,lap_number,lap_duration,stint_number,compound,tyre_age
0,1,97.211,1,MEDIUM,0.0
1,2,91.773,1,MEDIUM,1.0
24,25,96.063,1,MEDIUM,24.0
25,26,101.491,1,MEDIUM,25.0
26,27,119.169,2,INTERMEDIATE,0.0
27,28,101.176,2,INTERMEDIATE,1.0
36,37,98.869,2,INTERMEDIATE,10.0
37,38,95.963,2,INTERMEDIATE,11.0
38,39,118.311,3,HARD,0.0
39,40,90.056,3,HARD,1.0


In [16]:
def add_stint_info(driver_laps, driver_stints):
    driver_laps = driver_laps.copy()

    driver_laps["stint_number"] = pd.NA
    driver_laps["compound"] = pd.NA
    driver_laps["tyre_age"] = pd.NA

    for _, stint in driver_stints.iterrows():

        mask = (
            (driver_laps["lap_number"] >= stint["lap_start"]) &
            (driver_laps["lap_number"] <= stint["lap_end"])
        )

        driver_laps.loc[mask, "stint_number"] = stint["stint_number"]

        driver_laps.loc[mask, "compound"] = stint["compound"]

        driver_laps.loc[mask, "tyre_age"] = (
            driver_laps.loc[mask, "lap_number"] - stint["lap_start"]
        )

    return driver_laps

In [17]:
ver_laps_test = add_stint_info(ver_laps, ver_stints)

display(
    ver_laps_test[
        [
            "lap_number",
            "lap_duration",
            "stint_number",
            "compound",
            "tyre_age"
        ]
    ].head(30)
)

,lap_number,lap_duration,stint_number,compound,tyre_age
0,1,97.211,1,MEDIUM,0.0
1,2,91.773,1,MEDIUM,1.0
2,3,92.200,1,MEDIUM,2.0
3,4,91.933,1,MEDIUM,3.0
4,5,91.778,1,MEDIUM,4.0
5,6,91.711,1,MEDIUM,5.0
6,7,91.935,1,MEDIUM,6.0
7,8,92.038,1,MEDIUM,7.0
8,9,91.883,1,MEDIUM,8.0
9,10,91.661,1,MEDIUM,9.0


In [18]:
all_driver_laps = []

for driver_number in laps["driver_number"].unique():

    driver_laps = laps[
        laps["driver_number"] == driver_number
    ]
    driver_stints = stints[
        stints["driver_number"] == driver_number
    ]
    driver_result = add_stint_info(
        driver_laps,
        driver_stints
    )
    all_driver_laps.append(driver_result)

lap_performance = pd.concat(
    all_driver_laps,
    ignore_index=True
)

print("Shape:", lap_performance.shape)

display(
    lap_performance[
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "stint_number",
            "compound",
            "tyre_age"
        ]
    ].head(104)
)

Shape: (962, 19)


,driver_number,lap_number,lap_duration,stint_number,compound,tyre_age
0,1,1,97.211,1,MEDIUM,0.0
1,1,2,91.773,1,MEDIUM,1.0
2,1,3,92.200,1,MEDIUM,2.0
3,1,4,91.933,1,MEDIUM,3.0
4,1,5,91.778,1,MEDIUM,4.0
...,...,...,...,...,...,...
99,2,48,91.201,3,SOFT,9.0
100,2,49,91.175,3,SOFT,10.0
101,2,50,91.249,3,SOFT,11.0
102,2,51,91.662,3,SOFT,12.0


In [19]:
print("Total laps:", len(lap_performance))

print(
    "\nMissing stint information:"
)

print(
    lap_performance[
        ["stint_number", "compound", "tyre_age"]
    ].isna().sum()
)

Total laps: 962

Missing stint information:
stint_number    2
compound        2
tyre_age        2
dtype: int64


In [20]:
unmapped_laps = lap_performance[
    lap_performance["stint_number"].isna()
]

display(
    unmapped_laps[
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "date_start"
        ]
    ]
)

,driver_number,lap_number,lap_duration,date_start
207,10,1,NaN,2024-07-07 14:03:12.540000+00:00
858,63,34,NaN,2024-07-07 14:56:04.669000+00:00


In [21]:
affected_drivers = [10, 63]

display(
    stints[
        stints["driver_number"].isin(affected_drivers)
    ][
        [
            "driver_number",
            "stint_number",
            "lap_start",
            "lap_end",
            "compound",
            "tyre_age_at_start"
        ]
    ]
)

,driver_number,stint_number,lap_start,lap_end,compound,tyre_age_at_start
12,10,1,NaN,NaN,MEDIUM,0
57,63,1,1.0,27.0,MEDIUM,0
58,63,2,28.0,33.0,INTERMEDIATE,0


In [22]:
mapped_laps = lap_performance["stint_number"].notna().sum()
unmapped_laps = lap_performance["stint_number"].isna().sum()

print("Total laps:", len(lap_performance))
print("Mapped laps:", mapped_laps)
print("Unmapped laps:", unmapped_laps)

print(
    f"\nMapping coverage: "
    f"{mapped_laps / len(lap_performance) * 100:.2f}%"
)

Total laps: 962
Mapped laps: 960
Unmapped laps: 2

Mapping coverage: 99.79%


In [23]:
output_file = processed_dir / "lap_performance.parquet"

lap_performance.to_parquet(
    output_file,
    index=False
)

print(f"Saved to: {output_file}")
print(f"Shape: {lap_performance.shape}")

Saved to: ..\data\processed\9558\lap_performance.parquet
Shape: (962, 19)


In [24]:
lap_performance_check = pd.read_parquet(
    output_file
)

print("Original shape:", lap_performance.shape)
print("Reloaded shape:", lap_performance_check.shape)

print("\nColumns:")
print(lap_performance_check.columns.tolist())

print("\nMissing values:")
print(lap_performance_check.isna().sum())

Original shape: (962, 19)
Reloaded shape: (962, 19)

Columns:
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age']

Missing values:
meeting_key            0
session_key            0
driver_number          0
lap_number             0
date_start             1
duration_sector_1      3
duration_sector_2      2
duration_sector_3      2
i1_speed             134
i2_speed               2
is_pit_out_lap         0
lap_duration           3
segments_sector_1      1
segments_sector_2      1
segments_sector_3      1
st_speed               2
stint_number           2
compound               2
tyre_age               2
dtype: int64


In [25]:
lap_performance["lap_duration"].describe()

count    959.000000
mean      96.833269
std        7.612357
min       88.293000
25%       91.717000
50%       93.219000
75%      101.995000
max      128.459000
Name: lap_duration, dtype: float64

In [26]:
slowest_laps = (
    lap_performance.sort_values("lap_duration", ascending=False)
    [
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "is_pit_out_lap",
            "stint_number",
            "compound",
            "tyre_age"
        ]
    ]
    .head(15)
)
slowest_laps

,driver_number,lap_number,lap_duration,is_pit_out_lap,stint_number,compound,tyre_age
544,23,28,128.459,True,2,INTERMEDIATE,0.0
606,24,38,126.729,True,5,SOFT,0.0
896,77,38,126.628,True,3,SOFT,0.0
79,2,28,125.981,True,2,INTERMEDIATE,0.0
588,24,20,125.955,True,3,INTERMEDIATE,0.0
255,11,48,125.507,True,5,SOFT,0.0
885,77,27,125.367,True,2,INTERMEDIATE,0.0
440,20,28,125.299,True,2,INTERMEDIATE,0.0
595,24,27,124.515,True,4,INTERMEDIATE,0.0
285,14,28,124.427,True,2,INTERMEDIATE,0.0


In [27]:
fastest_laps = (
    lap_performance.sort_values("lap_duration")
    [
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "is_pit_out_lap",
            "stint_number",
            "compound",
            "tyre_age"
        ]
    ]
    .head(15)
)
fastest_laps

,driver_number,lap_number,lap_duration,is_pit_out_lap,stint_number,compound,tyre_age
824,55,52,88.293,False,4,SOFT,1.0
960,81,51,88.748,False,3,MEDIUM,12.0
47,1,48,88.952,False,3,HARD,9.0
956,81,47,89.049,False,3,MEDIUM,8.0
51,1,52,89.089,False,3,HARD,13.0
42,1,43,89.105,False,3,HARD,4.0
957,81,48,89.124,False,3,MEDIUM,9.0
44,1,45,89.137,False,3,HARD,6.0
953,81,44,89.221,False,3,MEDIUM,5.0
954,81,45,89.222,False,3,MEDIUM,6.0


In [28]:
pace_laps = lap_performance[
    lap_performance["lap_duration"].notna() & (lap_performance["is_pit_out_lap"] == False)
].copy()

print("Total laps:", len(lap_performance))
print("Pace laps:", len(pace_laps))
print("Excluded laps:", len(lap_performance) - len(pace_laps))

Total laps: 962
Pace laps: 914
Excluded laps: 48


In [29]:
pace_laps["lap_duration"].describe()

count    914.000000
mean      95.646827
std        5.479802
min       88.293000
25%       91.674250
50%       93.091500
75%      100.648250
max      118.710000
Name: lap_duration, dtype: float64

In [30]:
pace_laps.groupby("driver_number")["lap_duration"].count().sort_values()

driver_number
63    32
11    45
31    46
24    46
16    48
3     49
77    49
55    49
18    50
14    50
2     50
20    50
23    50
22    50
4     50
1     50
44    50
27    50
81    50
Name: lap_duration, dtype: int64

In [31]:
pace_laps.groupby("compound")["lap_duration"].agg(
    ["count", "mean", "median", "std"]
).sort_values("count", ascending=False)

,count,mean,median,std
compound,,,,
MEDIUM,500,94.108638,92.7565,4.272523
INTERMEDIATE,211,103.021882,103.0290,2.589708
SOFT,162,91.892235,91.1995,2.393977
HARD,41,91.285902,89.8720,2.513330


In [32]:
pace_laps.groupby("compound").agg(
    laps=("lap_number", "count"),
    first_lap=("lap_number", "min"),
    last_lap=("lap_number", "max"),
    drivers=("driver_number", "nunique")
)

,laps,first_lap,last_lap,drivers
compound,,,,
HARD,41,2,52,3
INTERMEDIATE,211,21,39,19
MEDIUM,500,1,52,19
SOFT,162,1,52,13


In [33]:
compound_by_lap = (
    pace_laps
    .groupby(["lap_number", "compound"])
    .size()
    .unstack(fill_value=0)
)

compound_by_lap.head(10)

compound,HARD,INTERMEDIATE,MEDIUM,SOFT
lap_number,,,,
1,0,0,16,2
2,1,0,16,2
3,1,0,16,2
4,1,0,16,2
5,1,0,16,2
6,1,0,16,2
7,1,0,16,2
8,1,0,16,2
9,1,0,16,2


In [34]:
tyre_age_pace = (
    pace_laps.groupby(["compound", "tyre_age"])["lap_duration"].agg(["count","median","mean"]).reset_index()
)
tyre_age_pace.head(20)

,compound,tyre_age,count,median,mean
0,HARD,1.0,3,90.4750,91.157667
1,HARD,2.0,3,89.8720,90.605333
2,HARD,3.0,3,89.8720,90.638000
3,HARD,4.0,3,89.7390,90.318333
4,HARD,5.0,3,89.5890,90.403667
5,HARD,6.0,3,89.5880,90.679333
6,HARD,7.0,3,89.5530,90.834333
7,HARD,8.0,3,89.4150,90.542000
8,HARD,9.0,3,89.4390,90.366667
9,HARD,10.0,3,89.5600,90.495000


In [35]:
weather = pd.read_parquet(
    processed_dir / "weather.parquet"
)

weather[[
    "date",
    "track_temperature",
    "air_temperature",
    "humidity",
    "rainfall",
    "wind_speed"
]].head()

weather[["date","rainfall"]].loc[
    weather["rainfall"] == True
]

,date,rainfall
5,2024-07-07 13:10:02.120000+00:00,1
6,2024-07-07 13:11:02.102000+00:00,1
7,2024-07-07 13:12:02.096000+00:00,1
8,2024-07-07 13:13:02.111000+00:00,1
9,2024-07-07 13:14:02.125000+00:00,1
10,2024-07-07 13:15:02.145000+00:00,1
11,2024-07-07 13:16:02.153000+00:00,1
12,2024-07-07 13:17:02.146000+00:00,1
13,2024-07-07 13:18:02.160000+00:00,1
14,2024-07-07 13:19:02.145000+00:00,1


In [36]:
ver_laps = lap_performance[
    lap_performance["driver_number"] == 1
].copy()

ver_laps[
    [
        "lap_number",
        "date_start",
        "lap_duration",
        "compound",
        "tyre_age",
        "is_pit_out_lap"
    ]
].tail(15)

,lap_number,date_start,lap_duration,compound,tyre_age,is_pit_out_lap
37,38,2024-07-07 15:02:45.144000+00:00,95.963,INTERMEDIATE,11.0,False
38,39,2024-07-07 15:04:21.179000+00:00,118.311,HARD,0.0,True
39,40,2024-07-07 15:06:19.420000+00:00,90.056,HARD,1.0,False
40,41,2024-07-07 15:07:49.775000+00:00,89.604,HARD,2.0,False
41,42,2024-07-07 15:09:19.236000+00:00,89.472,HARD,3.0,False
42,43,2024-07-07 15:10:48.675000+00:00,89.105,HARD,4.0,False
43,44,2024-07-07 15:12:17.680000+00:00,89.372,HARD,5.0,False
44,45,2024-07-07 15:13:47.096000+00:00,89.137,HARD,6.0,False
45,46,2024-07-07 15:15:16.171000+00:00,89.543,HARD,7.0,False
46,47,2024-07-07 15:16:45.763000+00:00,89.411,HARD,8.0,False


In [37]:
ver_laps.groupby(
    ["stint_number", "compound"]
).agg(
    start_lap=("lap_number", "min"),
    end_lap=("lap_number", "max"),
    laps=("lap_number", "count"),
    median_lap=("lap_duration", "median"),
    best_lap=("lap_duration", "min")
).reset_index()

,stint_number,compound,start_lap,end_lap,laps,median_lap,best_lap
0,1,MEDIUM,1,26,26,92.0905,91.661
1,2,INTERMEDIATE,27,38,12,102.0615,95.963
2,3,HARD,39,52,14,89.4270,88.952


In [38]:
position = pd.read_parquet(
    processed_dir / "position.parquet"
)

ver_position = position[
    position["driver_number"] == 1
].copy()

ver_position.head(10)

,date,session_key,driver_number,position,meeting_key
0,2024-07-07 13:05:02.926000+00:00,9558,1,4,1240
30,2024-07-07 14:03:31.245000+00:00,9558,1,3,1240
32,2024-07-07 14:03:31.254000+00:00,9558,1,4,1240
44,2024-07-07 14:03:37.140000+00:00,9558,1,3,1240
83,2024-07-07 14:25:59.612000+00:00,9558,1,4,1240
87,2024-07-07 14:29:05.679000+00:00,9558,1,5,1240
93,2024-07-07 14:31:13.305000+00:00,9558,1,4,1240
95,2024-07-07 14:31:15.050000+00:00,9558,1,5,1240
154,2024-07-07 14:45:51.799000+00:00,9558,1,4,1240
176,2024-07-07 14:47:33.122000+00:00,9558,1,3,1240


In [39]:
race_state = lap_performance[
    [
        "driver_number",
        "lap_number",
        "date_start",
        "lap_duration",
        "compound",
        "tyre_age",
        "stint_number",
        "is_pit_out_lap"
    ]
].copy()

race_state = race_state.sort_values(
    ["driver_number", "lap_number"]
).reset_index(drop=True)

position = position.sort_values(
    ["driver_number", "date"]
).reset_index(drop=True)

race_state = race_state.sort_values(
    ["driver_number", "date_start"]
).reset_index(drop=True)

race_state.head(10)


,driver_number,lap_number,date_start,lap_duration,compound,tyre_age,stint_number,is_pit_out_lap
0,1,1,2024-07-07 14:03:12.540000+00:00,97.211,MEDIUM,0.0,1,False
1,1,2,2024-07-07 14:04:49.750000+00:00,91.773,MEDIUM,1.0,1,False
2,1,3,2024-07-07 14:06:21.461000+00:00,92.200,MEDIUM,2.0,1,False
3,1,4,2024-07-07 14:07:53.675000+00:00,91.933,MEDIUM,3.0,1,False
4,1,5,2024-07-07 14:09:25.527000+00:00,91.778,MEDIUM,4.0,1,False
5,1,6,2024-07-07 14:10:57.353000+00:00,91.711,MEDIUM,5.0,1,False
6,1,7,2024-07-07 14:12:29.113000+00:00,91.935,MEDIUM,6.0,1,False
7,1,8,2024-07-07 14:14:01.043000+00:00,92.038,MEDIUM,7.0,1,False
8,1,9,2024-07-07 14:15:33.002000+00:00,91.883,MEDIUM,8.0,1,False
9,1,10,2024-07-07 14:17:04.948000+00:00,91.661,MEDIUM,9.0,1,False


In [40]:
race_state_valid = race_state[
    race_state["date_start"].notna()
].copy()

race_state_missing_time = race_state[
    race_state["date_start"].isna()
].copy()

print("Rows with valid timestamps:", len(race_state_valid))
print("Rows with missing timestamps:", len(race_state_missing_time))

Rows with valid timestamps: 961
Rows with missing timestamps: 1


In [41]:
race_state_valid= pd.merge_asof(
    race_state_valid.sort_values("date_start"),
    position[
        [
            "driver_number",
            "date",
            "position"
        ]
    ].sort_values("date"),
    left_on="date_start",
    right_on="date",
    by="driver_number",
    direction="backward"
)
race_state.head(10)

,driver_number,lap_number,date_start,lap_duration,compound,tyre_age,stint_number,is_pit_out_lap
0,1,1,2024-07-07 14:03:12.540000+00:00,97.211,MEDIUM,0.0,1,False
1,1,2,2024-07-07 14:04:49.750000+00:00,91.773,MEDIUM,1.0,1,False
2,1,3,2024-07-07 14:06:21.461000+00:00,92.200,MEDIUM,2.0,1,False
3,1,4,2024-07-07 14:07:53.675000+00:00,91.933,MEDIUM,3.0,1,False
4,1,5,2024-07-07 14:09:25.527000+00:00,91.778,MEDIUM,4.0,1,False
5,1,6,2024-07-07 14:10:57.353000+00:00,91.711,MEDIUM,5.0,1,False
6,1,7,2024-07-07 14:12:29.113000+00:00,91.935,MEDIUM,6.0,1,False
7,1,8,2024-07-07 14:14:01.043000+00:00,92.038,MEDIUM,7.0,1,False
8,1,9,2024-07-07 14:15:33.002000+00:00,91.883,MEDIUM,8.0,1,False
9,1,10,2024-07-07 14:17:04.948000+00:00,91.661,MEDIUM,9.0,1,False


In [42]:
print("race_state_valid shape:", race_state_valid.shape)

print("\nColumns:")
print(race_state_valid.columns.tolist())

print("\nMissing positions:")
print(race_state_valid["position"].isna().sum())

race_state_valid shape: (961, 10)

Columns:
['driver_number', 'lap_number', 'date_start', 'lap_duration', 'compound', 'tyre_age', 'stint_number', 'is_pit_out_lap', 'date', 'position']

Missing positions:
0


In [43]:
race_state = pd.concat(
    [race_state_valid, race_state_missing_time],
    ignore_index=True
)

race_state = race_state.sort_values(
    ["driver_number", "lap_number"]
).reset_index(drop=True)

print("Race state shape:", race_state.shape)
print("Missing timestamps:", race_state["date_start"].isna().sum())
print("Missing positions:", race_state["position"].isna().sum())

Race state shape: (962, 10)
Missing timestamps: 1
Missing positions: 1


In [44]:
race_state = race_state.drop(columns=["date"])

print(race_state.columns.tolist())

['driver_number', 'lap_number', 'date_start', 'lap_duration', 'compound', 'tyre_age', 'stint_number', 'is_pit_out_lap', 'position']


In [45]:
weather = weather.sort_values("date").reset_index(drop=True)
race_state = race_state.sort_values("date_start").reset_index(drop=True)
print(weather[["date", "rainfall"]].head())

race_state_valid = race_state[
    race_state["date_start"].notna()
].copy()

race_state_missing_time = race_state[
    race_state["date_start"].isna()
].copy()

race_state_valid = pd.merge_asof(
    race_state_valid,
    weather,
    left_on="date_start",
    right_on="date",
    direction="backward"
)

print("Shape:", race_state_valid.shape)
print(race_state_valid.columns.tolist())
print(race_state_valid.isna().sum())

                              date  rainfall
0 2024-07-07 13:05:02.119000+00:00         0
1 2024-07-07 13:06:02.098000+00:00         0
2 2024-07-07 13:07:02.122000+00:00         0
3 2024-07-07 13:08:02.127000+00:00         0
4 2024-07-07 13:09:02.125000+00:00         0
Shape: (961, 19)
['driver_number', 'lap_number', 'date_start', 'lap_duration', 'compound', 'tyre_age', 'stint_number', 'is_pit_out_lap', 'position', 'date', 'session_key', 'meeting_key', 'track_temperature', 'wind_speed', 'rainfall', 'humidity', 'pressure', 'air_temperature', 'wind_direction']
driver_number        0
lap_number           0
date_start           0
lap_duration         2
compound             2
tyre_age             2
stint_number         2
is_pit_out_lap       0
position             0
date                 0
session_key          0
meeting_key          0
track_temperature    0
wind_speed           0
rainfall             0
humidity             0
pressure             0
air_temperature      0
wind_direction       

In [46]:
race_state_valid["weather_age_seconds"] = (
  race_state_valid["date_start"] - race_state_valid["date"]
 ).dt.total_seconds()

race_state_valid["weather_age_seconds"].describe()

race_state_valid[
    ["date_start", "date", "weather_age_seconds", "rainfall"]
].head(10)

,date_start,date,weather_age_seconds,rainfall
0,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,0
1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,0
2,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,0
3,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,0
4,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,0
5,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,0
6,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,0
7,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,0
8,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,0
9,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:03:02.496000+00:00,10.044,0


In [47]:
race_state_valid["weather_age_seconds"].describe()

print(
    "Maximum weather age:",
    race_state_valid["weather_age_seconds"].max(),
    "seconds"
)

print(
    "Rain-affected driver-laps:",
    race_state_valid["rainfall"].sum()
)

Maximum weather age: 59.981 seconds
Rain-affected driver-laps: 407


In [48]:
race_state = pd.concat(
    [race_state_valid, race_state_missing_time],
    ignore_index=True
)

race_state = race_state.sort_values(
    ["driver_number", "lap_number"]
).reset_index(drop=True)

print("Shape:", race_state.shape)
print("Missing timestamps:", race_state["date_start"].isna().sum())
print("Missing positions:", race_state["position"].isna().sum())

race_state.head(10)

race_state[
    [
        "driver_number",
        "lap_number",
        "date_start",
        "position",
        "compound",
        "tyre_age",
        "track_temperature",
        "air_temperature",
        "rainfall",
        "weather_age_seconds"
    ]
].head(10)

Shape: (962, 20)
Missing timestamps: 1
Missing positions: 1


,driver_number,lap_number,date_start,position,compound,tyre_age,track_temperature,air_temperature,rainfall,weather_age_seconds
0,1,1,2024-07-07 14:03:12.540000+00:00,4.0,MEDIUM,0.0,33.0,16.1,0.0,10.044
1,1,2,2024-07-07 14:04:49.750000+00:00,3.0,MEDIUM,1.0,33.0,16.2,0.0,47.246
2,1,3,2024-07-07 14:06:21.461000+00:00,3.0,MEDIUM,2.0,33.0,16.3,0.0,18.957
3,1,4,2024-07-07 14:07:53.675000+00:00,3.0,MEDIUM,3.0,24.3,16.3,0.0,51.154
4,1,5,2024-07-07 14:09:25.527000+00:00,3.0,MEDIUM,4.0,24.0,16.2,0.0,22.990
5,1,6,2024-07-07 14:10:57.353000+00:00,3.0,MEDIUM,5.0,24.2,16.3,0.0,54.814
6,1,7,2024-07-07 14:12:29.113000+00:00,3.0,MEDIUM,6.0,25.4,16.3,0.0,26.554
7,1,8,2024-07-07 14:14:01.043000+00:00,3.0,MEDIUM,7.0,25.8,16.4,0.0,58.476
8,1,9,2024-07-07 14:15:33.002000+00:00,3.0,MEDIUM,8.0,26.1,16.5,0.0,30.422
9,1,10,2024-07-07 14:17:04.948000+00:00,3.0,MEDIUM,9.0,25.7,16.8,0.0,2.385


In [49]:
race_control = pd.read_parquet(
    processed_dir / "race_control.parquet"
)

race_control.head()

race_control[
    [
        "date",
        "category",
        "flag",
        "scope",
        "message"
    ]
].head(20)

race_control["category"].value_counts()
race_control["flag"].value_counts(dropna=False)

flag
BLUE         57
NaN          38
CLEAR        12
GREEN         2
CHEQUERED     1
Name: count, dtype: int64

In [50]:
race_state["lap_end"] = (
    race_state
    .groupby("driver_number")["date_start"]
    .shift(-1)
)

race_state[
    [
        "driver_number",
        "lap_number",
        "date_start",
        "lap_end"
    ]
].head(10)

,driver_number,lap_number,date_start,lap_end
0,1,1,2024-07-07 14:03:12.540000+00:00,2024-07-07 14:04:49.750000+00:00
1,1,2,2024-07-07 14:04:49.750000+00:00,2024-07-07 14:06:21.461000+00:00
2,1,3,2024-07-07 14:06:21.461000+00:00,2024-07-07 14:07:53.675000+00:00
3,1,4,2024-07-07 14:07:53.675000+00:00,2024-07-07 14:09:25.527000+00:00
4,1,5,2024-07-07 14:09:25.527000+00:00,2024-07-07 14:10:57.353000+00:00
5,1,6,2024-07-07 14:10:57.353000+00:00,2024-07-07 14:12:29.113000+00:00
6,1,7,2024-07-07 14:12:29.113000+00:00,2024-07-07 14:14:01.043000+00:00
7,1,8,2024-07-07 14:14:01.043000+00:00,2024-07-07 14:15:33.002000+00:00
8,1,9,2024-07-07 14:15:33.002000+00:00,2024-07-07 14:17:04.948000+00:00
9,1,10,2024-07-07 14:17:04.948000+00:00,2024-07-07 14:18:36.554000+00:00


In [51]:
race_control["category"].value_counts(dropna=False)

race_control[
    ["date", "category", "flag", "scope", "message"]
].sort_values("date").to_string(index=False)

(
    race_control[
        ["category", "flag", "scope", "message"]
    ]
    .drop_duplicates()
    .sort_values(["category", "flag"], na_position="last")
)

,category,flag,scope,message
4,DRS,NaN,NaN,DRS DISABLED
7,DRS,NaN,NaN,DRS ENABLED
15,FLAG,BLUE,DRIVER,WAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:3...
16,FLAG,BLUE,DRIVER,WAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:3...
18,FLAG,BLUE,DRIVER,WAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:3...
...,...,...,...,...
74,OTHER,NaN,NaN,TRACK SURFACE SLIPPERY IN TRACK SECTOR 10
78,OTHER,NaN,NaN,CAR 23 (ALB) TIME 1:45.202 DELETED - TRACK LIM...
87,OTHER,NaN,NaN,CAR 23 (ALB) TIME 1:43.075 DELETED - TRACK LIM...
5,SESSIONSTATUS,NaN,NaN,SESSION STARTED


In [52]:
other_events = (
    race_control[
        race_control["category"] == "OTHER"
    ][
        [
            "date",
            "driver_number",
            "scope",
            "message"
        ]
    ].sort_values("date")
)

print(other_events.to_string(index=False))

other_events["message"].value_counts().to_string()



                     date  driver_number scope                                                                                                                             message
2024-07-07 13:09:42+00:00           <NA>   NaN                                                                                                                 AWNINGS MAY BE USED
2024-07-07 13:30:00+00:00           <NA>   NaN                                                                                                                     PIT EXIT CLOSED
2024-07-07 13:45:18+00:00           <NA>   NaN                                                                                                     RISK OF RAIN FOR F1 RACE IS 60%
2024-07-07 14:06:28+00:00           <NA>   NaN                                           TURN 3 INCIDENT INVOLVING CAR 27 (HUL) NOTED - LEAVING THE TRACK AND GAINING AN ADVANTAGE
2024-07-07 14:10:19+00:00           <NA>   NaN FIA STEWARDS: TURN 3 INCIDENT INVOLVING CAR 27 (HUL) REVIE

'message\nTRACK SURFACE SLIPPERY IN TRACK SECTOR 17                                                                                              9\nLOW GRIP CONDITIONS                                                                                                                    2\nNORMAL GRIP CONDITIONS                                                                                                                 2\nAWNINGS MAY BE USED                                                                                                                    1\nPIT EXIT CLOSED                                                                                                                        1\nRISK OF RAIN FOR F1 RACE IS 60%                                                                                                        1\nTURN 3 INCIDENT INVOLVING CAR 27 (HUL) NOTED - LEAVING THE TRACK AND GAINING AN ADVANTAGE                                              1\nFIA STEWARDS: TURN 3 INC

In [53]:
race_control[
    race_control["flag"].notna()
    & (race_control["flag"] != "BLUE")
][
    [
        "date",
        "category",
        "flag",
        "scope",
        "driver_number",
        "message"
    ]
].to_string(index=False)

'                     date category      flag  scope  driver_number                     message\n2024-07-07 13:20:00+00:00     FLAG     GREEN  TRACK           <NA> GREEN LIGHT - PIT EXIT OPEN\n2024-07-07 14:03:13+00:00     FLAG     GREEN  TRACK           <NA> GREEN LIGHT - PIT EXIT OPEN\n2024-07-07 14:35:07+00:00     FLAG     CLEAR SECTOR           <NA>    CLEAR IN TRACK SECTOR 17\n2024-07-07 14:35:19+00:00     FLAG     CLEAR SECTOR           <NA>    CLEAR IN TRACK SECTOR 17\n2024-07-07 14:35:28+00:00     FLAG     CLEAR SECTOR           <NA>    CLEAR IN TRACK SECTOR 17\n2024-07-07 14:36:01+00:00     FLAG     CLEAR SECTOR           <NA>    CLEAR IN TRACK SECTOR 17\n2024-07-07 14:36:01+00:00     FLAG     CLEAR SECTOR           <NA>    CLEAR IN TRACK SECTOR 11\n2024-07-07 14:36:24+00:00     FLAG     CLEAR SECTOR           <NA>    CLEAR IN TRACK SECTOR 17\n2024-07-07 14:36:28+00:00     FLAG     CLEAR SECTOR           <NA>    CLEAR IN TRACK SECTOR 17\n2024-07-07 14:36:50+00:00     FLAG     

In [54]:
race_state_rc = race_state.copy()

race_state_rc["race_control_count"] = 0
race_state_rc["drs_event"] = False
race_state_rc["blue_flag"] = False
race_state_rc["track_surface_slippery"] = False
race_state_rc["low_grip"] = False
race_state_rc["normal_grip"] = False

for _, event in race_control.iterrows():

    event_time = event["date"]

    if pd.isna(event_time):
        continue

    mask = (
        race_state_rc["date_start"].notna()
        & race_state_rc["lap_end"].notna()
        & (race_state_rc["date_start"] <= event_time)
        & (event_time < race_state_rc["lap_end"])
    )

    if not mask.any():
        continue

    category = str(event["category"]).upper()
    message = str(event["message"]).upper()
    flag = str(event["flag"]).upper()

    race_state_rc.loc[mask, "race_control_count"] += 1


    if category == "DRS":
        race_state_rc.loc[mask, "drs_event"] = True


    elif category == "FLAG" and flag == "BLUE":
        driver = event["driver_number"]

        driver_mask = mask & (
            race_state_rc["driver_number"] == driver
        )

        race_state_rc.loc[
            driver_mask,
            "blue_flag"
        ] = True

    elif category == "OTHER":

        if "SLIPPERY" in message:
            race_state_rc.loc[
                mask,
                "track_surface_slippery"
            ] = True

        if "LOW GRIP" in message:
            race_state_rc.loc[
                mask,
                "low_grip"
            ] = True

        if "NORMAL GRIP" in message:
            race_state_rc.loc[
                mask,
                "normal_grip"
            ] = True

In [55]:
race_state_rc[
    [
        "race_control_count",
        "drs_event",
        "blue_flag",
        "track_surface_slippery",
        "low_grip",
        "normal_grip"
    ]
].sum()

race_control_count        1893
drs_event                   93
blue_flag                   29
track_surface_slippery      76
low_grip                    38
normal_grip                 37
dtype: int64

In [56]:
race_state_rc[
    race_state_rc["race_control_count"] > 0
][
    [
        "driver_number",
        "lap_number",
        "date_start",
        "race_control_count",
        "drs_event",
        "blue_flag",
        "track_surface_slippery",
        "low_grip",
        "normal_grip"
    ]
].head(5)

,driver_number,lap_number,date_start,race_control_count,drs_event,blue_flag,track_surface_slippery,low_grip,normal_grip
0,1,1,2024-07-07 14:03:12.540000+00:00,3,True,False,False,False,False
2,1,3,2024-07-07 14:06:21.461000+00:00,1,False,False,False,False,False
4,1,5,2024-07-07 14:09:25.527000+00:00,1,False,False,False,False,False
16,1,17,2024-07-07 14:27:48.466000+00:00,1,False,False,False,False,False
18,1,19,2024-07-07 14:31:00.053000+00:00,4,True,False,False,True,False


In [57]:
race_control[
    race_control["message"].str.contains(
        "SLIPPERY|LOW GRIP|NORMAL GRIP",
        case=False,
        na=False
    )
][
    [
        "date",
        "category",
        "scope",
        "message"
    ]
].sort_values("date").to_string(index=False)

'                     date category scope                                   message\n2024-07-07 14:31:23+00:00    OTHER   NaN                       LOW GRIP CONDITIONS\n2024-07-07 14:35:05+00:00    OTHER   NaN TRACK SURFACE SLIPPERY IN TRACK SECTOR 17\n2024-07-07 14:35:16+00:00    OTHER   NaN TRACK SURFACE SLIPPERY IN TRACK SECTOR 17\n2024-07-07 14:35:27+00:00    OTHER   NaN TRACK SURFACE SLIPPERY IN TRACK SECTOR 17\n2024-07-07 14:35:57+00:00    OTHER   NaN TRACK SURFACE SLIPPERY IN TRACK SECTOR 11\n2024-07-07 14:36:00+00:00    OTHER   NaN TRACK SURFACE SLIPPERY IN TRACK SECTOR 17\n2024-07-07 14:36:23+00:00    OTHER   NaN TRACK SURFACE SLIPPERY IN TRACK SECTOR 17\n2024-07-07 14:36:27+00:00    OTHER   NaN TRACK SURFACE SLIPPERY IN TRACK SECTOR 17\n2024-07-07 14:36:48+00:00    OTHER   NaN TRACK SURFACE SLIPPERY IN TRACK SECTOR 17\n2024-07-07 14:37:02+00:00    OTHER   NaN TRACK SURFACE SLIPPERY IN TRACK SECTOR 17\n2024-07-07 14:39:11+00:00    OTHER   NaN                    NORMAL GRIP CON

In [58]:
race_control[
    race_control["driver_number"].notna()
][
    [
        "date",
        "category",
        "scope",
        "driver_number",
        "flag",
        "message"
    ]
].sort_values("date").to_string(index=False)


'                     date category  scope  driver_number flag                                            message\n2024-07-07 14:34:24+00:00     FLAG DRIVER             24 BLUE WAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:34:23\n2024-07-07 14:34:28+00:00     FLAG DRIVER             24 BLUE WAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:34:27\n2024-07-07 14:34:33+00:00     FLAG DRIVER             24 BLUE WAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:34:32\n2024-07-07 14:34:41+00:00     FLAG DRIVER             24 BLUE WAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:34:40\n2024-07-07 14:37:32+00:00     FLAG DRIVER             24 BLUE WAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:37:32\n2024-07-07 14:37:40+00:00     FLAG DRIVER             24 BLUE WAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:37:40\n2024-07-07 14:37:50+00:00     FLAG DRIVER             24 BLUE WAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:37:49\n2024-07-07 14:37:58+00:00     FLAG DRIVER             24 BLUE WAVED BLUE FLAG FOR CAR 2

In [59]:
race_control[
    race_control["driver_number"].notna()
]["message"].dropna().drop_duplicates().to_string(index=False)

'WAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:34:23\nWAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:34:27\nWAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:34:32\nWAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:34:40\nWAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:37:32\nWAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:37:40\nWAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:37:49\nWAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:37:57\nWAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:38:58\nWAVED BLUE FLAG FOR CAR 11 (PER) TIMED AT 15:40:24\nWAVED BLUE FLAG FOR CAR 24 (ZHO) TIMED AT 15:40:33\nWAVED BLUE FLAG FOR CAR 11 (PER) TIMED AT 15:40:36\nWAVED BLUE FLAG FOR CAR 16 (LEC) TIMED AT 15:41:00\nWAVED BLUE FLAG FOR CAR 11 (PER) TIMED AT 15:41:00\nWAVED BLUE FLAG FOR CAR 16 (LEC) TIMED AT 15:41:22\nWAVED BLUE FLAG FOR CAR 11 (PER) TIMED AT 15:41:49\nWAVED BLUE FLAG FOR CAR 16 (LEC) TIMED AT 15:42:28\nWAVED BLUE FLAG FOR CAR 31 (OCO) TIMED AT 15:44:15\nWAVED BLUE FLAG FOR CAR 31 (OCO) TIMED AT 15:44:35\nWAVED BLUE 

In [60]:
race_control[
    [
        "date",
        "category",
        "scope",
        "driver_number",
        "flag",
        "message"
    ]
].sort_values("date").to_string(index=False)

'                            date      category  scope  driver_number      flag                                                                                                                             message\n       2024-07-07 13:09:42+00:00         OTHER    NaN           <NA>       NaN                                                                                                                 AWNINGS MAY BE USED\n       2024-07-07 13:20:00+00:00          FLAG  TRACK           <NA>     GREEN                                                                                                         GREEN LIGHT - PIT EXIT OPEN\n       2024-07-07 13:30:00+00:00         OTHER    NaN           <NA>       NaN                                                                                                                     PIT EXIT CLOSED\n       2024-07-07 13:45:18+00:00         OTHER    NaN           <NA>       NaN                                                                         

In [61]:
race_state_final = race_state.copy()

race_state_final = race_state_final.sort_values(
    ["driver_number", "lap_number"]
).reset_index(drop=True)

race_state_final.shape

(962, 21)

In [62]:
print("RACE STATE")
print("shape:", race_state.shape)
print("date_start dtype:", race_state["date_start"].dtype)
print("date_start nulls:", race_state["date_start"].isna().sum())

print("\nRACE CONTROL")
print("shape:", race_control.shape)
print("date dtype:", race_control["date"].dtype)
print("date nulls:", race_control["date"].isna().sum())

print("\nDATE_START SAMPLE")
print(race_state["date_start"].head())

print("\nRACE CONTROL DATE SAMPLE")
print(race_control["date"].head())

RACE STATE
shape: (962, 21)
date_start dtype: datetime64[us, UTC]
date_start nulls: 1

RACE CONTROL
shape: (110, 11)
date dtype: datetime64[us, UTC]
date nulls: 0

DATE_START SAMPLE
0   2024-07-07 14:03:12.540000+00:00
1   2024-07-07 14:04:49.750000+00:00
2   2024-07-07 14:06:21.461000+00:00
3   2024-07-07 14:07:53.675000+00:00
4   2024-07-07 14:09:25.527000+00:00
Name: date_start, dtype: datetime64[us, UTC]

RACE CONTROL DATE SAMPLE
0   2024-07-07 13:09:42+00:00
1   2024-07-07 13:20:00+00:00
2   2024-07-07 13:30:00+00:00
3   2024-07-07 13:45:18+00:00
4   2024-07-07 13:57:16+00:00
Name: date, dtype: datetime64[us, UTC]


In [63]:
print("\nSORT CHECK")

race_state_valid = race_state[
    race_state["date_start"].notna()
].copy()

print(
    "race_state_valid sorted:",
    race_state_valid["date_start"].is_monotonic_increasing
)

print(
    "race_control sorted:",
    race_control["date"].is_monotonic_increasing
)


SORT CHECK
race_state_valid sorted: False
race_control sorted: True


In [64]:
print("\nDUPLICATES")

print(
    "race_state duplicate timestamps:",
    race_state_valid["date_start"].duplicated().sum()
)

print(
    "race_control duplicate timestamps:",
    race_control["date"].duplicated().sum()
)


DUPLICATES
race_state duplicate timestamps: 18
race_control duplicate timestamps: 2


In [65]:
race_state_valid = race_state[
    race_state["date_start"].notna()
].copy()

race_state_missing_time = race_state[
    race_state["date_start"].isna()
].copy()

race_state_valid = race_state_valid.sort_values(
    "date_start",
    kind="stable"
).reset_index(drop=True)

print("Valid rows:", len(race_state_valid))
print("Missing-time rows:", len(race_state_missing_time))
print("Sorted:", race_state_valid["date_start"].is_monotonic_increasing)

Valid rows: 961
Missing-time rows: 1
Sorted: True


In [66]:
drs_events = race_control[
    race_control["category"].eq("DRS")
][["date", "message"]].copy()

drs_events = drs_events.sort_values(
    "date",
    kind="stable"
).reset_index(drop=True)

drs_events["drs_available"] = (
    drs_events["message"]
    .str.upper()
    .eq("DRS ENABLED")
)

print(drs_events)
print("\nSorted:", drs_events["date"].is_monotonic_increasing)

                       date       message  drs_available
0 2024-07-07 13:57:16+00:00  DRS DISABLED          False
1 2024-07-07 14:04:48+00:00   DRS ENABLED           True
2 2024-07-07 14:31:20+00:00  DRS DISABLED          False
3 2024-07-07 14:38:48+00:00   DRS ENABLED           True
4 2024-07-07 14:41:45+00:00  DRS DISABLED          False
5 2024-07-07 15:08:39+00:00   DRS ENABLED           True

Sorted: True


In [67]:
test_drs = pd.merge_asof(
    race_state_valid,
    drs_events[["date", "drs_available"]],
    left_on="date_start",
    right_on="date",
    direction="backward"
)

print(test_drs.shape)
print(
    test_drs[
        ["driver_number", "lap_number", "date_start",
         "drs_available"]
    ].head(20).to_string(index=False)
)

(961, 23)
 driver_number  lap_number                       date_start  drs_available
             1           1 2024-07-07 14:03:12.540000+00:00          False
             2           1 2024-07-07 14:03:12.540000+00:00          False
             3           1 2024-07-07 14:03:12.540000+00:00          False
             4           1 2024-07-07 14:03:12.540000+00:00          False
            10           1 2024-07-07 14:03:12.540000+00:00          False
            14           1 2024-07-07 14:03:12.540000+00:00          False
            16           1 2024-07-07 14:03:12.540000+00:00          False
            18           1 2024-07-07 14:03:12.540000+00:00          False
            20           1 2024-07-07 14:03:12.540000+00:00          False
            22           1 2024-07-07 14:03:12.540000+00:00          False
            23           1 2024-07-07 14:03:12.540000+00:00          False
            24           1 2024-07-07 14:03:12.540000+00:00          False
            27 

In [68]:
race_state_valid = test_drs.copy()

print(race_state_valid.shape)
print(race_state_valid["drs_available"].value_counts(dropna=False))

(961, 23)
drs_available
True     573
False    388
Name: count, dtype: int64


In [69]:
grip_events = race_control[
    race_control["message"].str.upper().isin([
        "LOW GRIP CONDITIONS",
        "NORMAL GRIP CONDITIONS"
    ])
][["date", "message"]].copy()

grip_events = grip_events.sort_values(
    "date",
    kind="stable"
).reset_index(drop=True)

grip_events["grip_condition"] = (
    grip_events["message"]
    .str.upper()
    .map({
        "LOW GRIP CONDITIONS": "LOW",
        "NORMAL GRIP CONDITIONS": "NORMAL"
    })
)

print(grip_events)

                       date                 message grip_condition
0 2024-07-07 14:31:23+00:00     LOW GRIP CONDITIONS            LOW
1 2024-07-07 14:39:11+00:00  NORMAL GRIP CONDITIONS         NORMAL
2 2024-07-07 14:41:47+00:00     LOW GRIP CONDITIONS            LOW
3 2024-07-07 15:08:51+00:00  NORMAL GRIP CONDITIONS         NORMAL


In [70]:
test_grip = pd.merge_asof(
    race_state_valid.sort_values(
        "date_start",
        kind="stable"
    ),
    grip_events[["date", "grip_condition"]],
    left_on="date_start",
    right_on="date",
    direction="backward"
)

test_grip = test_grip.drop(columns=["date"])

print(test_grip.shape)
print(
    test_grip["grip_condition"].value_counts(
        dropna=False
    )
)

(961, 24)
grip_condition
LOW       377
NaN       356
NORMAL    228
Name: count, dtype: int64


In [71]:
race_state_valid = test_grip.copy()

In [72]:
race_state_valid["blue_flag"] = False

blue_flags = race_control[
    (race_control["category"].eq("FLAG")) &
    (race_control["scope"].eq("DRIVER")) &
    (race_control["flag"].eq("BLUE"))
][["date", "driver_number"]].copy()

blue_flags = blue_flags.dropna(
    subset=["date", "driver_number"]
).sort_values(
    "date",
    kind="stable"
).reset_index(drop=True)

print(blue_flags)

                        date  driver_number
0  2024-07-07 14:34:24+00:00             24
1  2024-07-07 14:34:28+00:00             24
2  2024-07-07 14:34:33+00:00             24
3  2024-07-07 14:34:41+00:00             24
4  2024-07-07 14:37:32+00:00             24
5  2024-07-07 14:37:40+00:00             24
6  2024-07-07 14:37:50+00:00             24
7  2024-07-07 14:37:58+00:00             24
8  2024-07-07 14:38:59+00:00             24
9  2024-07-07 14:40:24+00:00             11
10 2024-07-07 14:40:33+00:00             24
11 2024-07-07 14:40:36+00:00             11
12 2024-07-07 14:41:00+00:00             16
13 2024-07-07 14:41:01+00:00             11
14 2024-07-07 14:41:22+00:00             16
15 2024-07-07 14:41:49+00:00             11
16 2024-07-07 14:42:29+00:00             16
17 2024-07-07 14:44:15+00:00             31
18 2024-07-07 14:44:36+00:00             31
19 2024-07-07 14:44:44+00:00             31
20 2024-07-07 14:44:52+00:00             31
21 2024-07-07 14:45:49+00:00    

In [73]:
for _, event in blue_flags.iterrows():

    mask = (
        race_state_valid["driver_number"].eq(
            event["driver_number"]
        )
        & race_state_valid["lap_end"].notna()
        & race_state_valid["date_start"].le(
            event["date"]
        )
        & race_state_valid["lap_end"].gt(
            event["date"]
        )
    )

    race_state_valid.loc[mask, "blue_flag"] = True

In [74]:
print(
    "Blue-flag laps:",
    race_state_valid["blue_flag"].sum()
)

Blue-flag laps: 29


In [75]:
race_state_final = pd.concat(
    [
        race_state_valid,
        race_state_missing_time
    ],
    ignore_index=True
)

race_state_final = race_state_final.sort_values(
    ["driver_number", "lap_number"]
).reset_index(drop=True)

In [76]:
print("Shape:", race_state_final.shape)

print(
    "\nMissing values:"
)

print(
    race_state_final[
        [
            "date_start",
            "position",
            "drs_available",
            "grip_condition",
            "blue_flag"
        ]
    ].isna().sum()
)

print(
    "\nDuplicate driver/lap:",
    race_state_final.duplicated(
        ["driver_number", "lap_number"]
    ).sum()
)

Shape: (962, 26)

Missing values:
date_start          1
position            1
drs_available       1
grip_condition    357
blue_flag           1
dtype: int64

Duplicate driver/lap: 0


In [77]:
print(
    race_state_final[
        (race_state_final["driver_number"] == 1) &
        (race_state_final["lap_number"].between(20, 45))
    ][
        [
            "lap_number",
            "date_start",
            "compound",
            "tyre_age",
            "drs_available",
            "grip_condition",
            "blue_flag"
        ]
    ].to_string(index=False)
)

 lap_number                       date_start     compound tyre_age drs_available grip_condition blue_flag
         20 2024-07-07 14:32:42.464000+00:00       MEDIUM     19.0         False            LOW     False
         21 2024-07-07 14:34:19.699000+00:00       MEDIUM     20.0         False            LOW     False
         22 2024-07-07 14:35:53.074000+00:00       MEDIUM     21.0         False            LOW     False
         23 2024-07-07 14:37:25.010000+00:00       MEDIUM     22.0         False            LOW     False
         24 2024-07-07 14:38:57.270000+00:00       MEDIUM     23.0          True            LOW     False
         25 2024-07-07 14:40:30.085000+00:00       MEDIUM     24.0          True         NORMAL     False
         26 2024-07-07 14:42:06.267000+00:00       MEDIUM     25.0         False            LOW     False
         27 2024-07-07 14:43:47.712000+00:00 INTERMEDIATE      0.0         False            LOW     False
         28 2024-07-07 14:45:46.850000+00:00 I

In [78]:
print(race_state_valid.columns.tolist())

['driver_number', 'lap_number', 'date_start', 'lap_duration', 'compound', 'tyre_age', 'stint_number', 'is_pit_out_lap', 'position', 'date_x', 'session_key', 'meeting_key', 'track_temperature', 'wind_speed', 'rainfall', 'humidity', 'pressure', 'air_temperature', 'wind_direction', 'weather_age_seconds', 'lap_end', 'date_y', 'drs_available', 'grip_condition', 'blue_flag']


In [79]:
race_state_final = pd.concat(
    [
        race_state_valid,
        race_state_missing_time
    ],
    ignore_index=True
)

race_state_final = race_state_final.sort_values(
    ["driver_number", "lap_number"]
).reset_index(drop=True)

In [80]:
print("Shape:", race_state_final.shape)

print(
    "Duplicate driver/lap:",
    race_state_final.duplicated(
        ["driver_number", "lap_number"]
    ).sum()
)

print(
    "Missing date_start:",
    race_state_final["date_start"].isna().sum()
)

print(
    "Missing position:",
    race_state_final["position"].isna().sum()
)

Shape: (962, 26)
Duplicate driver/lap: 0
Missing date_start: 1
Missing position: 1


In [81]:
race_state_final = race_state_final.drop(
    columns=["date_x", "date_y", "lap_end"]
)

print(race_state_final.columns.tolist())
print(race_state_final.shape)

race_state_path = processed_dir / "race_state.parquet"

race_state_final.to_parquet(
    race_state_path,
    index=False
)

print("Saved:", race_state_path)

race_state_check = pd.read_parquet(race_state_path)

print("Shape:", race_state_check.shape)
print("Columns:", race_state_check.columns.tolist())

['driver_number', 'lap_number', 'date_start', 'lap_duration', 'compound', 'tyre_age', 'stint_number', 'is_pit_out_lap', 'position', 'session_key', 'meeting_key', 'track_temperature', 'wind_speed', 'rainfall', 'humidity', 'pressure', 'air_temperature', 'wind_direction', 'weather_age_seconds', 'drs_available', 'grip_condition', 'blue_flag', 'date']
(962, 23)
Saved: ..\data\processed\9558\race_state.parquet
Shape: (962, 23)
Columns: ['driver_number', 'lap_number', 'date_start', 'lap_duration', 'compound', 'tyre_age', 'stint_number', 'is_pit_out_lap', 'position', 'session_key', 'meeting_key', 'track_temperature', 'wind_speed', 'rainfall', 'humidity', 'pressure', 'air_temperature', 'wind_direction', 'weather_age_seconds', 'drs_available', 'grip_condition', 'blue_flag', 'date']


## 7. Telemetry Aggregation

In [82]:
from pathlib import Path
import pandas as pd

processed_dir = Path("../data/processed/9558")

car_data_1 = pd.read_parquet(processed_dir / "car_data" / "1.parquet")

print("Shape:", car_data_1.shape)
print("\nColumns:")
print(car_data_1.columns.tolist())

print("\nData types:")
print(car_data_1.dtypes)

print("\nFirst 5 rows:")
display(car_data_1.head())

Shape: (33181, 10)

Columns:
['date', 'session_key', 'meeting_key', 'driver_number', 'n_gear', 'drs', 'brake', 'rpm', 'speed', 'throttle']

Data types:
date             datetime64[us, UTC]
session_key                    int64
meeting_key                    int64
driver_number                  int64
n_gear                         int64
drs                            int64
brake                          int64
rpm                            int64
speed                          int64
throttle                       int64
dtype: object

First 5 rows:


,date,session_key,meeting_key,driver_number,n_gear,drs,brake,rpm,speed,throttle
0,2024-07-07 13:04:59.448000+00:00,9558,1240,1,0,0,0,0,0,0
1,2024-07-07 13:04:59.728000+00:00,9558,1240,1,0,0,0,0,0,0
2,2024-07-07 13:05:00.088000+00:00,9558,1240,1,0,0,0,0,0,0
3,2024-07-07 13:05:00.288000+00:00,9558,1240,1,0,0,0,0,0,0
4,2024-07-07 13:05:00.488000+00:00,9558,1240,1,0,0,0,0,0,0


In [83]:
print("Telemetry start:", car_data_1["date"].min())
print("Telemetry end:", car_data_1["date"].max())

print("\nMissing values:")
print(car_data_1.isna().sum())

print("\nDuplicate timestamps:", car_data_1["date"].duplicated().sum())

print("\nTime differences between samples:")
print(car_data_1["date"].diff().describe())

Telemetry start: 2024-07-07 13:04:59.448000+00:00
Telemetry end: 2024-07-07 15:30:50.677000+00:00

Missing values:
date             0
session_key      0
meeting_key      0
driver_number    0
n_gear           0
drs              0
brake            0
rpm              0
speed            0
throttle         0
dtype: int64

Duplicate timestamps: 0

Time differences between samples:
count                     33180
mean     0 days 00:00:00.263750
std      0 days 00:00:00.091241
min      0 days 00:00:00.159000
25%      0 days 00:00:00.200000
50%      0 days 00:00:00.240000
75%      0 days 00:00:00.320000
max      0 days 00:00:01.360000
Name: date, dtype: object


In [84]:
telemetry = car_data_1.copy()

telemetry = telemetry.sort_values("date").reset_index(drop=True)

ver_laps = race_state[
    race_state["driver_number"] == 1
][[
    "driver_number",
    "lap_number",
    "date_start",
    "lap_end"
]].copy()

ver_laps = ver_laps.dropna(
    subset=["date_start", "lap_end"]
)

telemetry_lap = pd.merge_asof(
    telemetry,
    ver_laps.sort_values("date_start"),
    left_on="date",
    right_on="date_start",
    direction="backward"
)

telemetry_lap = telemetry_lap[
    telemetry_lap["date"] < telemetry_lap["lap_end"]
].copy()

print("Telemetry samples assigned to laps:", len(telemetry_lap))
print("Unique laps assigned:", telemetry_lap["lap_number"].nunique())

display(
    telemetry_lap[
        ["date", "lap_number", "speed", "throttle", "brake", "rpm", "n_gear", "drs"]
    ].head(10)
)

Telemetry samples assigned to laps: 18452
Unique laps assigned: 51


,date,lap_number,speed,throttle,brake,rpm,n_gear,drs
13227,2024-07-07 14:03:12.660000+00:00,1.0,0,17,100,10149,1,1
13228,2024-07-07 14:03:12.980000+00:00,1.0,0,17,0,9323,1,1
13229,2024-07-07 14:03:13.380000+00:00,1.0,12,17,0,6523,1,1
13230,2024-07-07 14:03:13.619000+00:00,1.0,18,18,0,4843,1,1
13231,2024-07-07 14:03:13.859000+00:00,1.0,24,20,0,3946,1,1
13232,2024-07-07 14:03:14.259000+00:00,1.0,42,37,0,5433,1,1
13233,2024-07-07 14:03:14.619000+00:00,1.0,59,53,0,6998,1,1
13234,2024-07-07 14:03:14.819000+00:00,1.0,68,57,0,8398,1,1
13235,2024-07-07 14:03:14.979000+00:00,1.0,72,67,0,9238,1,1
13236,2024-07-07 14:03:15.299000+00:00,1.0,84,77,0,10249,1,1


In [85]:
telemetry_lap_features = (
    telemetry_lap
    .groupby(["driver_number_x", "lap_number"], as_index=False)
    .agg(
        speed_mean=("speed", "mean"),
        speed_max=("speed", "max"),
        throttle_mean=("throttle", "mean"),
        brake_pct=("brake", lambda x: (x > 0).mean() * 100),
        rpm_mean=("rpm", "mean"),
        rpm_max=("rpm", "max"),
        gear_mean=("n_gear", "mean"),
        gear_max=("n_gear", "max"),
        drs_pct=("drs", lambda x: (x > 0).mean() * 100),
        telemetry_samples=("date", "count")
    )
)

print("Shape:", telemetry_lap_features.shape)

display(telemetry_lap_features.head(10))

Shape: (51, 12)


,driver_number_x,lap_number,speed_mean,speed_max,throttle_mean,brake_pct,rpm_mean,rpm_max,gear_mean,gear_max,drs_pct,telemetry_samples
0,1,1.0,213.200000,302,69.858333,16.388889,10503.941667,12060,5.177778,8,99.444444,360
1,1,2.0,228.064327,317,70.783626,17.251462,10471.020468,12245,5.684211,8,31.286550,342
2,1,3.0,225.429775,298,69.415730,17.134831,10413.275281,11919,5.620787,8,0.000000,356
3,1,4.0,227.217647,300,70.970588,14.705882,10489.517647,12087,5.655882,8,0.000000,340
4,1,5.0,225.373887,300,70.753709,15.133531,10485.326409,11997,5.593472,8,0.000000,337
5,1,6.0,227.766382,300,71.424501,15.384615,10470.245014,11943,5.649573,8,0.000000,351
6,1,7.0,228.675287,299,72.577586,15.804598,10455.543103,11968,5.715517,8,0.000000,348
7,1,8.0,224.193642,300,70.361272,17.052023,10423.630058,11853,5.560694,8,0.000000,346
8,1,9.0,229.000000,301,73.918539,13.764045,10558.497191,12039,5.662921,8,0.000000,356
9,1,10.0,227.344729,300,70.418803,15.669516,10453.968661,11908,5.663818,8,0.000000,351


In [86]:
telemetry_coverage = telemetry_lap_features[
    ["driver_number_x", "lap_number", "telemetry_samples"]
].copy()

print("Telemetry lap features shape:", telemetry_lap_features.shape)

print("\nTelemetry samples per lap:")
print(telemetry_coverage["telemetry_samples"].describe())

print("\nLaps with fewer than 100 samples:")
display(
    telemetry_coverage[
        telemetry_coverage["telemetry_samples"] < 100
    ]
)

print("\nDuplicate driver-lap combinations:",
      telemetry_coverage.duplicated(
          subset=["driver_number_x", "lap_number"]
      ).sum())

Telemetry lap features shape: (51, 12)

Telemetry samples per lap:
count     51.000000
mean     361.803922
std       25.684252
min      329.000000
25%      345.000000
50%      354.000000
75%      372.500000
max      458.000000
Name: telemetry_samples, dtype: float64

Laps with fewer than 100 samples:


,driver_number_x,lap_number,telemetry_samples



Duplicate driver-lap combinations: 0


In [87]:
telemetry_lap_features = telemetry_lap_features.rename(
    columns={"driver_number_x": "driver_number"}
)

print(telemetry_lap_features.columns.tolist())

['driver_number', 'lap_number', 'speed_mean', 'speed_max', 'throttle_mean', 'brake_pct', 'rpm_mean', 'rpm_max', 'gear_mean', 'gear_max', 'drs_pct', 'telemetry_samples']


In [88]:
race_state = race_state.merge(
    telemetry_lap_features,
    on = ["driver_number", "lap_number"],
    how="left"
)

print("Shape:", race_state.shape)

display(race_state.head())

Shape: (962, 31)


,driver_number,lap_number,date_start,lap_duration,compound,tyre_age,stint_number,is_pit_out_lap,position,date,...,speed_mean,speed_max,throttle_mean,brake_pct,rpm_mean,rpm_max,gear_mean,gear_max,drs_pct,telemetry_samples
0,1,1,2024-07-07 14:03:12.540000+00:00,97.211,MEDIUM,0.0,1,False,4.0,2024-07-07 14:03:02.496000+00:00,...,213.200000,302.0,69.858333,16.388889,10503.941667,12060.0,5.177778,8.0,99.444444,360.0
1,1,2,2024-07-07 14:04:49.750000+00:00,91.773,MEDIUM,1.0,1,False,3.0,2024-07-07 14:04:02.504000+00:00,...,228.064327,317.0,70.783626,17.251462,10471.020468,12245.0,5.684211,8.0,31.286550,342.0
2,1,3,2024-07-07 14:06:21.461000+00:00,92.200,MEDIUM,2.0,1,False,3.0,2024-07-07 14:06:02.504000+00:00,...,225.429775,298.0,69.415730,17.134831,10413.275281,11919.0,5.620787,8.0,0.000000,356.0
3,1,4,2024-07-07 14:07:53.675000+00:00,91.933,MEDIUM,3.0,1,False,3.0,2024-07-07 14:07:02.521000+00:00,...,227.217647,300.0,70.970588,14.705882,10489.517647,12087.0,5.655882,8.0,0.000000,340.0
4,1,5,2024-07-07 14:09:25.527000+00:00,91.778,MEDIUM,4.0,1,False,3.0,2024-07-07 14:09:02.537000+00:00,...,225.373887,300.0,70.753709,15.133531,10485.326409,11997.0,5.593472,8.0,0.000000,337.0


In [89]:
[x for x in race_state.columns if "_x" in x or "_y" in x]

race_state[[
    "speed_mean_x", "speed_mean_y",
    "speed_max_x", "speed_max_y"
]].head()

KeyError: "None of [Index(['speed_mean_x', 'speed_mean_y', 'speed_max_x', 'speed_max_y'], dtype='str')] are in the [columns]"

In [ ]:
duplicate_cols = [
    col for col in race_state.columns
    if col.endswith("_x") or col.endswith("_y")
]

print(duplicate_cols)

['speed_mean_x', 'speed_max_x', 'throttle_mean_x', 'brake_pct_x', 'rpm_mean_x', 'rpm_max_x', 'gear_mean_x', 'gear_max_x', 'drs_pct_x', 'telemetry_samples_x', 'speed_mean_y', 'speed_max_y', 'throttle_mean_y', 'brake_pct_y', 'rpm_mean_y', 'rpm_max_y', 'gear_mean_y', 'gear_max_y', 'drs_pct_y', 'telemetry_samples_y']


In [ ]:
print(race_state.columns.tolist())

for col in race_state.columns:
    if "speed_mean" in col or "telemetry" in col:
        print(repr(col))

print(race_state.shape)

['driver_number', 'lap_number', 'date_start', 'lap_duration', 'compound', 'tyre_age', 'stint_number', 'is_pit_out_lap', 'position', 'date', 'session_key', 'meeting_key', 'track_temperature', 'wind_speed', 'rainfall', 'humidity', 'pressure', 'air_temperature', 'wind_direction', 'weather_age_seconds', 'lap_end', 'speed_mean', 'speed_max', 'throttle_mean', 'brake_pct', 'rpm_mean', 'rpm_max', 'gear_mean', 'gear_max', 'drs_pct', 'telemetry_samples']
'speed_mean'
'telemetry_samples'
(962, 31)


In [ ]:
print(
    race_state[
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "compound",
            "tyre_age",
            "position",
            "speed_mean",
            "throttle_mean",
            "brake_pct",
            "drs_pct",
            "telemetry_samples"
        ]
    ].head()
)

   driver_number  lap_number  lap_duration compound tyre_age  position  \
0              1           1        97.211   MEDIUM      0.0       4.0   
1              1           2        91.773   MEDIUM      1.0       3.0   
2              1           3        92.200   MEDIUM      2.0       3.0   
3              1           4        91.933   MEDIUM      3.0       3.0   
4              1           5        91.778   MEDIUM      4.0       3.0   

   speed_mean  throttle_mean  brake_pct    drs_pct  telemetry_samples  
0  213.200000      69.858333  16.388889  99.444444              360.0  
1  228.064327      70.783626  17.251462  31.286550              342.0  
2  225.429775      69.415730  17.134831   0.000000              356.0  
3  227.217647      70.970588  14.705882   0.000000              340.0  
4  225.373887      70.753709  15.133531   0.000000              337.0  


In [ ]:
stint_summary = (
    race_state.groupby(
        ["driver_number", "stint_number", "compound"],
        dropna=False
    ).agg(
        laps=("lap_number", "count"),
        tyre_age_start=("tyre_age", "min"),
        tyre_age_end=("tyre_age", "max"),
        avg_lap_time=("lap_duration", "mean")
    ).reset_index()
)

display(stint_summary.head(20))

,driver_number,stint_number,compound,laps,tyre_age_start,tyre_age_end,avg_lap_time
0,1,1.0,MEDIUM,26,0.0,25.0,93.664308
1,1,2.0,INTERMEDIATE,12,0.0,11.0,102.784250
2,1,3.0,HARD,14,0.0,13.0,91.452929
3,2,1.0,MEDIUM,27,0.0,26.0,95.400926
4,2,2.0,INTERMEDIATE,11,0.0,10.0,105.629909
5,2,3.0,SOFT,14,0.0,13.0,92.763214
6,3,1.0,MEDIUM,26,0.0,25.0,95.304846
7,3,2.0,INTERMEDIATE,11,0.0,10.0,105.518091
8,3,3.0,SOFT,14,0.0,13.0,94.197429
9,4,1.0,MEDIUM,27,0.0,26.0,93.619963


In [97]:
degradation_df = race_state[
    race_state["compound"].notna()
    & race_state["tyre_age"].notna()
    & race_state["lap_duration"].notna()
    & (~race_state["is_pit_out_lap"])
].copy()
print(degradation_df.shape)

display(
    degradation_df[
        [
            "driver_number",
            "lap_number",
            "stint_number",
            "compound",
            "tyre_age",
            "lap_duration"
        ]
    ].head(20)
)

print(degradation_df["compound"].value_counts())

print(degradation_df.groupby("compound")["tyre_age"].agg(
    ["min", "max", "mean", "count"]
))

(914, 31)


,driver_number,lap_number,stint_number,compound,tyre_age,lap_duration
0,1,1,1,MEDIUM,0.0,97.211
1,1,2,1,MEDIUM,1.0,91.773
2,1,3,1,MEDIUM,2.0,92.200
3,1,4,1,MEDIUM,3.0,91.933
4,1,5,1,MEDIUM,4.0,91.778
5,1,6,1,MEDIUM,5.0,91.711
6,1,7,1,MEDIUM,6.0,91.935
7,1,8,1,MEDIUM,7.0,92.038
8,1,9,1,MEDIUM,8.0,91.883
9,1,10,1,MEDIUM,9.0,91.661


compound
MEDIUM          500
INTERMEDIATE    211
SOFT            162
HARD             41
Name: count, dtype: int64
              min   max      mean  count
compound                                
HARD          1.0  18.0  7.731707     41
INTERMEDIATE  1.0  12.0  5.421801    211
MEDIUM        0.0  27.0      11.6    500
SOFT          0.0  18.0  6.950617    162


In [98]:
age_pace = (
    degradation_df.groupby(["compound","tyre_age"])
    .agg(
        mean_lap_time=("lap_duration", "mean"),
        median_lap_time=("lap_duration", "median"),
        laps=("lap_duration", "count")
    ).reset_index()
)
display(age_pace.head(30))



,compound,tyre_age,mean_lap_time,median_lap_time,laps
0,HARD,1.0,91.157667,90.4750,3
1,HARD,2.0,90.605333,89.8720,3
2,HARD,3.0,90.638000,89.8720,3
3,HARD,4.0,90.318333,89.7390,3
4,HARD,5.0,90.403667,89.5890,3
5,HARD,6.0,90.679333,89.5880,3
6,HARD,7.0,90.834333,89.5530,3
7,HARD,8.0,90.542000,89.4150,3
8,HARD,9.0,90.366667,89.4390,3
9,HARD,10.0,90.495000,89.5600,3


In [99]:
display(
    age_pace[age_pace["compound"] == "MEDIUM"]
)

,compound,tyre_age,mean_lap_time,median_lap_time,laps
30,MEDIUM,0.0,100.169813,100.2580,16
31,MEDIUM,1.0,92.572917,92.7025,24
32,MEDIUM,2.0,92.425667,92.3380,24
33,MEDIUM,3.0,92.834083,92.3040,24
34,MEDIUM,4.0,93.248417,92.2730,24
35,MEDIUM,5.0,92.250783,92.0620,23
36,MEDIUM,6.0,92.568043,92.6620,23
37,MEDIUM,7.0,91.881409,92.2050,22
38,MEDIUM,8.0,91.895955,92.3085,22
39,MEDIUM,9.0,92.078909,92.4005,22


In [100]:
display(
    race_state[
        [
            "lap_number",
            "driver_number",
            "compound",
            "tyre_age",
            "lap_duration",
            "rainfall",
            "track_temperature"
        ]
    ]
    .sort_values("lap_number")
    .tail(40)
)

print(
    race_state.groupby("lap_number")["rainfall"]
    .mean()
    .loc[30:52]
)

,lap_number,driver_number,compound,tyre_age,lap_duration,rainfall,track_temperature
908,50,77,SOFT,12.0,92.320,0.0,25.5
770,50,44,SOFT,11.0,89.682,0.0,25.2
566,50,23,MEDIUM,11.0,89.828,0.0,25.4
462,50,20,SOFT,12.0,91.143,0.0,25.4
410,50,18,MEDIUM,10.0,90.030,0.0,25.4
514,50,22,SOFT,11.0,91.407,0.0,25.4
359,50,16,SOFT,12.0,90.488,0.0,25.5
204,50,4,SOFT,10.0,90.544,0.0,25.2
153,50,3,SOFT,12.0,94.150,0.0,25.4
257,50,11,SOFT,2.0,89.707,0.0,25.3


lap_number
30    1.000000
31    1.000000
32    1.000000
33    1.000000
34    1.000000
35    1.000000
36    1.000000
37    1.000000
38    0.833333
39    0.388889
40    0.000000
41    0.000000
42    0.000000
43    0.166667
44    0.555556
45    0.666667
46    0.000000
47    0.000000
48    0.000000
49    0.000000
50    0.000000
51    0.000000
52    0.000000
Name: rainfall, dtype: float64


In [101]:
dry_degradation = degradation_df[
    degradation_df["compound"].isin(["SOFT", "MEDIUM", "HARD"])
    & (degradation_df["rainfall"] == 0)
    & (degradation_df["tyre_age"] > 0)
    & (degradation_df["lap_duration"] < 110)
].copy()

print("Shape:", dry_degradation.shape)

display(
    dry_degradation[
        [
            "driver_number",
            "lap_number",
            "stint_number",
            "compound",
            "tyre_age",
            "lap_duration",
            "rainfall",
            "track_temperature"
        ]
    ].head(20)
)

print(
    dry_degradation["compound"].value_counts()
)

print(
    dry_degradation.groupby("compound")["tyre_age"].agg(["min","max","count"])
)

Shape: (520, 31)


,driver_number,lap_number,stint_number,compound,tyre_age,lap_duration,rainfall,track_temperature
1,1,2,1,MEDIUM,1.0,91.773,0.0,33.0
2,1,3,1,MEDIUM,2.0,92.200,0.0,33.0
3,1,4,1,MEDIUM,3.0,91.933,0.0,24.3
4,1,5,1,MEDIUM,4.0,91.778,0.0,24.0
5,1,6,1,MEDIUM,5.0,91.711,0.0,24.2
6,1,7,1,MEDIUM,6.0,91.935,0.0,25.4
7,1,8,1,MEDIUM,7.0,92.038,0.0,25.8
8,1,9,1,MEDIUM,8.0,91.883,0.0,26.1
9,1,10,1,MEDIUM,9.0,91.661,0.0,25.7
10,1,11,1,MEDIUM,10.0,91.810,0.0,25.6


compound
MEDIUM    337
SOFT      145
HARD       38
Name: count, dtype: int64
          min   max  count
compound                  
HARD      1.0  17.0     38
MEDIUM    1.0  17.0    337
SOFT      1.0  16.0    145


In [102]:
stint_pace = (
    dry_degradation
    .groupby(
        ["driver_number", "stint_number", "compound"]
    )
    .agg(
        laps=("lap_duration", "count"),
        first_lap_time=("lap_duration", "first"),
        last_lap_time=("lap_duration", "last"),
        avg_lap_time=("lap_duration", "mean"),
        tyre_age_start=("tyre_age", "min"),
        tyre_age_end=("tyre_age", "max")
    ).reset_index()
)

display(stint_pace.head(20))

long_stints = stint_pace[
    stint_pace["laps"] >= 5
].copy()

print("Long stints:", len(long_stints))

display(
    long_stints.sort_values(
        ["compound", "laps"],
        ascending=[True, False]
    ).head(30)
)

,driver_number,stint_number,compound,laps,first_lap_time,last_lap_time,avg_lap_time,tyre_age_start,tyre_age_end
0,1,1,MEDIUM,17,91.773,96.172,92.374059,1.0,17.0
1,1,3,HARD,12,90.056,89.089,89.407750,1.0,13.0
2,2,1,MEDIUM,17,93.435,94.900,93.257882,1.0,17.0
3,2,3,SOFT,11,91.353,92.088,91.073909,1.0,13.0
4,3,1,MEDIUM,17,93.709,96.827,93.563765,1.0,17.0
5,3,3,SOFT,12,93.138,94.814,92.029667,1.0,13.0
6,4,1,MEDIUM,17,92.188,94.125,92.063059,1.0,17.0
7,4,3,SOFT,11,89.733,90.913,90.168000,1.0,12.0
8,11,1,HARD,17,92.942,95.384,93.049882,1.0,17.0
9,11,4,MEDIUM,7,92.722,90.795,91.238286,1.0,9.0


Long stints: 37


,driver_number,stint_number,compound,laps,first_lap_time,last_lap_time,avg_lap_time,tyre_age_start,tyre_age_end
8,11,1,HARD,17,92.942,95.384,93.049882,1.0,17.0
1,1,3,HARD,12,90.056,89.089,89.407750,1.0,13.0
33,55,3,HARD,9,90.475,89.434,89.709667,1.0,10.0
0,1,1,MEDIUM,17,91.773,96.172,92.374059,1.0,17.0
2,2,1,MEDIUM,17,93.435,94.900,93.257882,1.0,17.0
4,3,1,MEDIUM,17,93.709,96.827,93.563765,1.0,17.0
6,4,1,MEDIUM,17,92.188,94.125,92.063059,1.0,17.0
11,14,1,MEDIUM,17,93.202,95.880,92.987118,1.0,17.0
13,16,1,MEDIUM,17,92.858,95.951,92.918176,1.0,17.0
15,18,1,MEDIUM,17,92.683,95.789,93.019765,1.0,17.0


In [103]:
stint_laps = dry_degradation.sort_values(
    ["driver_number", "stint_number", "tyre_age"]
).copy()

stint_laps["lap_time_change"] = (
    stint_laps
    .groupby(["driver_number", "stint_number"])["lap_duration"]
    .diff()
)

display(
    stint_laps[
        [
            "driver_number",
            "stint_number",
            "compound",
            "tyre_age",
            "lap_duration",
            "lap_time_change"
        ]
    ].head(30)
)

,driver_number,stint_number,compound,tyre_age,lap_duration,lap_time_change
1,1,1,MEDIUM,1.0,91.773,NaN
2,1,1,MEDIUM,2.0,92.200,0.427
3,1,1,MEDIUM,3.0,91.933,-0.267
4,1,1,MEDIUM,4.0,91.778,-0.155
5,1,1,MEDIUM,5.0,91.711,-0.067
6,1,1,MEDIUM,6.0,91.935,0.224
7,1,1,MEDIUM,7.0,92.038,0.103
8,1,1,MEDIUM,8.0,91.883,-0.155
9,1,1,MEDIUM,9.0,91.661,-0.222
10,1,1,MEDIUM,10.0,91.810,0.149


In [104]:
display(
    stint_laps[
        stint_laps["driver_number"] == 1
    ][
        [
            "lap_number",
            "stint_number",
            "compound",
            "tyre_age",
            "lap_duration",
            "lap_time_change"
        ]
    ]
)

,lap_number,stint_number,compound,tyre_age,lap_duration,lap_time_change
1,2,1,MEDIUM,1.0,91.773,NaN
2,3,1,MEDIUM,2.0,92.200,0.427
3,4,1,MEDIUM,3.0,91.933,-0.267
4,5,1,MEDIUM,4.0,91.778,-0.155
5,6,1,MEDIUM,5.0,91.711,-0.067
6,7,1,MEDIUM,6.0,91.935,0.224
7,8,1,MEDIUM,7.0,92.038,0.103
8,9,1,MEDIUM,8.0,91.883,-0.155
9,10,1,MEDIUM,9.0,91.661,-0.222
10,11,1,MEDIUM,10.0,91.810,0.149


In [105]:
import numpy as np
import pandas as pd

stint_slopes = []

for (driver, stint, compound), group in dry_degradation.groupby(
    ["driver_number", "stint_number", "compound"]
):
    
    # Convert only the columns needed for the calculation
    x = pd.to_numeric(group["tyre_age"], errors="coerce")
    y = pd.to_numeric(group["lap_duration"], errors="coerce")

    # Remove any invalid values
    valid = x.notna() & y.notna()
    x = x[valid]
    y = y[valid]

    # Need enough valid laps
    if len(x) < 6:
        continue

    slope = np.polyfit(x.to_numpy(), y.to_numpy(), 1)[0]

    stint_slopes.append({
        "driver_number": driver,
        "stint_number": stint,
        "compound": compound,
        "laps": len(x),
        "pace_slope_sec_per_lap": slope
    })

stint_slopes = pd.DataFrame(stint_slopes)

display(stint_slopes.head(20))

dry_degradation["tyre_age"] = pd.to_numeric(
    dry_degradation["tyre_age"],
    errors="coerce"
)

print(dry_degradation["tyre_age"].dtype)

,driver_number,stint_number,compound,laps,pace_slope_sec_per_lap
0,1,1,MEDIUM,17,0.152336
1,1,3,HARD,12,-0.037712
2,2,1,MEDIUM,17,0.060164
3,2,3,SOFT,11,0.096050
4,3,1,MEDIUM,17,0.110485
5,3,3,SOFT,12,0.133428
6,4,1,MEDIUM,17,0.052794
7,4,3,SOFT,11,0.119812
8,11,1,HARD,17,0.115451
9,11,4,MEDIUM,7,-0.280488


float64


In [106]:
slope_summary = (
    stint_slopes
    .groupby("compound")["pace_slope_sec_per_lap"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max"
    )
    .reset_index()
)

display(slope_summary)

,compound,count,mean,median,std,min,max
0,HARD,3,-0.005232,-0.037712,0.108164,-0.093434,0.115451
1,MEDIUM,22,0.049257,0.068002,0.092951,-0.280488,0.152336
2,SOFT,12,0.045069,0.038375,0.125637,-0.232964,0.219951


In [107]:
dry_age_pace = (
    dry_degradation.groupby(["compound", "tyre_age"]).agg(
        mean_lap_time=("lap_duration", "mean"),
        median_lap_time=("lap_duration","median"),
        laps=("lap_duration","count")
    ).reset_index()
)
display(dry_age_pace)

display(
    dry_age_pace[
        dry_age_pace["compound"] == "MEDIUM"
    ]
)

display(
    dry_age_pace[
        dry_age_pace["compound"] == "SOFT"
    ]
)

display(
    dry_age_pace[
        dry_age_pace["compound"] == "HARD"
    ]
)

,compound,tyre_age,mean_lap_time,median_lap_time,laps
0,HARD,1.0,91.157667,90.4750,3
1,HARD,2.0,90.605333,89.8720,3
2,HARD,3.0,90.638000,89.8720,3
3,HARD,4.0,90.318333,89.7390,3
4,HARD,5.0,90.811000,90.8110,2
5,HARD,6.0,91.450500,91.4505,2
6,HARD,7.0,90.834333,89.5530,3
7,HARD,8.0,90.542000,89.4150,3
8,HARD,9.0,90.366667,89.4390,3
9,HARD,10.0,90.495000,89.5600,3


,compound,tyre_age,mean_lap_time,median_lap_time,laps
17,MEDIUM,1.0,92.446174,92.6830,23
18,MEDIUM,2.0,92.183087,92.3070,23
19,MEDIUM,3.0,92.199391,92.0450,23
20,MEDIUM,4.0,92.292286,92.6070,21
21,MEDIUM,5.0,92.289278,92.6435,18
22,MEDIUM,6.0,92.374167,92.7070,18
23,MEDIUM,7.0,91.881409,92.2050,22
24,MEDIUM,8.0,91.895955,92.3085,22
25,MEDIUM,9.0,92.078909,92.4005,22
26,MEDIUM,10.0,91.855143,92.1230,21


,compound,tyre_age,mean_lap_time,median_lap_time,laps
34,SOFT,1.0,92.537286,92.5070,14
35,SOFT,2.0,91.592692,90.8780,13
36,SOFT,3.0,90.997667,90.4645,12
37,SOFT,4.0,90.914000,90.3105,12
38,SOFT,5.0,91.461857,91.1980,7
39,SOFT,6.0,92.243200,91.3780,5
40,SOFT,7.0,91.577727,90.9550,11
41,SOFT,8.0,91.397333,91.0565,12
42,SOFT,9.0,91.456500,91.1185,12
43,SOFT,10.0,91.468500,91.0945,12


,compound,tyre_age,mean_lap_time,median_lap_time,laps
0,HARD,1.0,91.157667,90.4750,3
1,HARD,2.0,90.605333,89.8720,3
2,HARD,3.0,90.638000,89.8720,3
3,HARD,4.0,90.318333,89.7390,3
4,HARD,5.0,90.811000,90.8110,2
5,HARD,6.0,91.450500,91.4505,2
6,HARD,7.0,90.834333,89.5530,3
7,HARD,8.0,90.542000,89.4150,3
8,HARD,9.0,90.366667,89.4390,3
9,HARD,10.0,90.495000,89.5600,3


In [108]:
ml_features = dry_degradation[
    [
        "driver_number",
        "lap_number",
        "position",
        "compound",
        "tyre_age",
        "track_temperature",
        "air_temperature",
        "humidity",
        "wind_speed",
        "rainfall",
        "lap_duration"
    ]
].copy()

print("Shape:", ml_features.shape)
print("\nDtypes:")
print(ml_features.dtypes)

display(ml_features.head())



Shape: (520, 11)

Dtypes:
driver_number          int64
lap_number             int64
position             float64
compound              object
tyre_age             float64
track_temperature    float64
air_temperature      float64
humidity             float64
wind_speed           float64
rainfall             float64
lap_duration         float64
dtype: object


,driver_number,lap_number,position,compound,tyre_age,track_temperature,air_temperature,humidity,wind_speed,rainfall,lap_duration
1,1,2,3.0,MEDIUM,1.0,33.0,16.2,63.0,1.6,0.0,91.773
2,1,3,3.0,MEDIUM,2.0,33.0,16.3,62.0,2.0,0.0,92.200
3,1,4,3.0,MEDIUM,3.0,24.3,16.3,61.0,1.9,0.0,91.933
4,1,5,3.0,MEDIUM,4.0,24.0,16.2,60.0,2.2,0.0,91.778
5,1,6,3.0,MEDIUM,5.0,24.2,16.3,60.0,2.4,0.0,91.711


In [109]:
baseline_lap_model = ml_features.copy()

X = baseline_lap_model.drop(columns=["lap_duration"])

y = baseline_lap_model["lap_duration"]

print("Drivers:", X["driver_number"].nunique())

display(
    X["driver_number"]
    .value_counts()
    .sort_index()
)

Drivers: 19


driver_number
1     29
2     28
3     29
4     28
11    26
14    28
16    29
18    27
20    29
22    28
23    28
24    26
27    28
31    26
44    29
55    27
63    17
77    29
81    29
Name: count, dtype: int64

In [110]:
categorical_features = ["driver_number", "compound"]

numerical_features = [
    "lap_number",
    "position",
    "tyre_age",
    "track_temperature",
    "air_temperature",
    "humidity",
    "wind_speed",
    "rainfall"
]



In [111]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

split_lap = baseline_lap_model["lap_number"].quantile(0.8)

train_data = baseline_lap_model[
    baseline_lap_model["lap_number"] > split_lap
].copy()

test_data = baseline_lap_model[
    baseline_lap_model["lap_number"] > split_lap
]

print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)
print("Split lap:", split_lap)

Train shape: (97, 11)
Test shape: (97, 11)
Split lap: 46.0


In [117]:
X_train = train_data.drop(columns=["lap_duration"])
y_train = train_data["lap_duration"]

X_test = test_data.drop(columns=["lap_duration"])
y_test = test_data["lap_duration"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (97, 10)
X_test: (97, 10)
y_train: (97,)
y_test: (97,)


In [118]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

models = {
    "Ridge": Ridge(alpha=1.0),
    
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),
    
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)

        ]
    )

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    results.append({
        "model":name,
        "MAE": mae,
        "RMSE":rmse
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results)

results_df = results_df.sort_values("MAE")

display(results_df)


best_model_name = results_df.iloc[0]["model"]

best_model = trained_models[best_model_name]

predictions = best_model.predict(X_test)

prediction_check = X_test.copy()

prediction_check["actual_lap_time"] = y_test.values
prediction_check["predicted_lap_time"] = predictions
prediction_check["error"] = (
    prediction_check["actual_lap_time"]
    - prediction_check["predicted_lap_time"]
)

display(
    prediction_check[
        [
            "driver_number",
            "lap_number",
            "compound",
            "tyre_age",
            "actual_lap_time",
            "predicted_lap_time",
            "error"
        ]
    ].head(20)
)

,model,MAE,RMSE
2,Gradient Boosting,0.112988,0.149362
1,Random Forest,0.219538,0.431533
0,Ridge,0.467101,0.825551


,driver_number,lap_number,compound,tyre_age,actual_lap_time,predicted_lap_time,error
46,1,47,HARD,8.0,89.411,89.296579,0.114421
47,1,48,HARD,9.0,88.952,89.153452,-0.201452
48,1,49,HARD,10.0,89.560,89.398181,0.161819
49,1,50,HARD,11.0,89.286,89.429902,-0.143902
50,1,51,HARD,12.0,89.443,89.398126,0.044874
51,1,52,HARD,13.0,89.089,89.280885,-0.191885
98,2,47,SOFT,8.0,91.250,91.078250,0.171750
99,2,48,SOFT,9.0,91.201,91.151186,0.049814
100,2,49,SOFT,10.0,91.175,91.151186,0.023814
101,2,50,SOFT,11.0,91.249,91.346923,-0.097923


In [119]:
best_model = trained_models["Gradient Boosting"]

feature_names = (
    best_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

importances = (
    best_model
    .named_steps["model"]
    .feature_importances_
)

feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
)

display(feature_importance)

,feature,importance
22,num__position,0.494041
23,num__tyre_age,0.216695
2,cat__driver_number_3,0.061443
13,cat__driver_number_31,0.032096
20,cat__compound_SOFT,0.027471
3,cat__driver_number_4,0.026696
24,num__track_temperature,0.023373
21,num__lap_number,0.020113
27,num__wind_speed,0.015980
25,num__air_temperature,0.015220


In [120]:
display(
    prediction_check[
        [
            "driver_number",
            "lap_number",
            "compound",
            "tyre_age",
            "actual_lap_time",
            "predicted_lap_time",
            "error"
        ]
    ]
    .sort_values("error", key=lambda x: x.abs(), ascending=False)
    .head(15)
)

,driver_number,lap_number,compound,tyre_age,actual_lap_time,predicted_lap_time,error
718,31,48,MEDIUM,9.0,95.931,95.418140,0.512860
960,81,51,MEDIUM,12.0,88.748,89.149400,-0.401400
959,81,50,MEDIUM,11.0,89.958,89.575169,0.382831
153,3,50,SOFT,12.0,94.150,93.782257,0.367743
719,31,49,MEDIUM,10.0,91.060,91.385599,-0.325599
309,14,52,MEDIUM,13.0,91.430,91.122280,0.307720
304,14,47,MEDIUM,8.0,89.710,90.006040,-0.296040
257,11,50,SOFT,2.0,89.707,89.964808,-0.257808
306,14,49,MEDIUM,10.0,90.700,90.460169,0.239831
563,23,47,MEDIUM,8.0,90.148,89.918004,0.229996


In [121]:
prediction_check["absolute_error"] = prediction_check["error"].abs()

driver_error = (
    prediction_check
    .groupby("driver_number")
    .agg(
        mae=("absolute_error", "mean"),
        max_error=("absolute_error", "max"),
        laps=("absolute_error", "count")
    )
    .sort_values("mae")
)

display(driver_error)

,mae,max_error,laps
driver_number,,,
20,0.049826,0.118789,6
44,0.052297,0.107983,6
18,0.058332,0.124169,6
16,0.058975,0.124138,5
55,0.063840,0.178378,5
2,0.070224,0.171750,6
27,0.070375,0.123425,6
4,0.081572,0.180306,6
22,0.092890,0.222046,6


In [122]:
compound_error = (
    prediction_check
    .groupby("compound")
    .agg(
        mae=("absolute_error", "mean"),
        max_error=("absolute_error", "max"),
        laps=("absolute_error", "count")
    )
    .sort_values("mae")
)

display(compound_error)

,mae,max_error,laps
compound,,,
SOFT,0.091406,0.367743,58
HARD,0.111159,0.201452,10
MEDIUM,0.156782,0.512860,29


In [123]:
prediction_check["absolute_error"] = prediction_check["error"].abs()

driver_error = (
    prediction_check.groupby("driver_number").agg(
        mae=("absolute_error", "mean"),
        max_error=("absolute_error","max"),
        laps=("absolute_error","count")
    ).sort_values("mae")
)
display(driver_error)



,mae,max_error,laps
driver_number,,,
20,0.049826,0.118789,6
44,0.052297,0.107983,6
18,0.058332,0.124169,6
16,0.058975,0.124138,5
55,0.063840,0.178378,5
2,0.070224,0.171750,6
27,0.070375,0.123425,6
4,0.081572,0.180306,6
22,0.092890,0.222046,6
